<a href="https://colab.research.google.com/github/kevinscaria/InstructABSA/blob/main/ATE_Training_%26_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Libraries

In [1]:
try:
    import google.colab
    from google.colab import drive

    drive.mount('/content/drive', force_remount=True)
    IN_COLAB = True
except:
    IN_COLAB = False

In [2]:
if IN_COLAB:
    !pip install transformers
    !pip install datasets
    !pip install evaluate
    !pip install sentencepiece

In [3]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '2'
import torch

print(torch.cuda.is_available())
print(torch.backends.mps.is_built())
if IN_COLAB:
    root_path = 'Enter drive path'
else:
    root_path = '../'

# use_mps = True if torch.has_mps else False
use_mps = torch.backends.mps.is_built()
os.chdir(root_path)

True
False


In [4]:
import warnings

warnings.filterwarnings('ignore')
import pandas as pd

from InstructABSA.data_prep import DatasetLoader
from InstructABSA.utils import T5Generator, T5Classifier
from instructions import InstructionsHandler

## Training

In [5]:
task_name = 'ate'
experiment_name = 'lapt2014_iabsa1'
model_checkpoint = './model/tk-instruct-base-def-pos'
print('Experiment Name: ', experiment_name)
model_out_path = './Models'
model_out_path = os.path.join(model_out_path, task_name, f"{model_checkpoint.replace('/', '')}-{experiment_name}")
print('Model output path: ', model_out_path)

Experiment Name:  lapt2014_iabsa1
Model output path:  ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1


In [6]:
# Load the data
id_train_file_path = './Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = './Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path, encoding='gbk')

id_val_file_path = './Dataset/SemEval14/Validation/Laptops_val.csv'
id_val_df = pd.read_csv(id_val_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set1()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(train_df_id=id_tr_df, test_df_id=id_te_df, val_df_id=id_val_df)
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_ate_format(loader.train_df_id, 'term', 'raw_text', 'aspectTerms',
                                                          instruct_handler.ate['bos_instruct1'],
                                                          instruct_handler.ate['eos_instruct'])
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_ate_format(loader.test_df_id, 'term', 'raw_text', 'aspectTerms',
                                                         instruct_handler.ate['bos_instruct1'],
                                                         instruct_handler.ate['eos_instruct'])
if loader.val_df_id is not None:
    loader.val_df_id = loader.create_data_in_ate_format(loader.val_df_id, 'term', 'raw_text', 'aspectTerms',
                                                        instruct_handler.ate['bos_instruct1'],
                                                        instruct_handler.ate['eos_instruct'])

In [7]:
# Create T5 utils object
t5_exp = T5Generator(model_checkpoint)

# Tokenize Dataset
id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Training arguments
training_args = {
    'output_dir': model_out_path,
    'evaluation_strategy': "epoch",
    'learning_rate': 5e-5,
    'lr_scheduler_type': 'cosine',
    'per_device_train_batch_size': 8,
    # 'per_device_train_batch_size': 1,
    'per_device_eval_batch_size': 16,
    # 'per_device_eval_batch_size': 2,
    'num_train_epochs': 4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'save_strategy': 'no',
    'load_best_model_at_end': False,
    'push_to_hub': False,
    'eval_accumulation_steps': 1,
    'predict_with_generate': True,
    # 'use_mps_device': use_mps
}

Map:   0%|          | 0/3045 [00:00<?, ? examples/s]

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Map:   0%|          | 0/201 [00:00<?, ? examples/s]

In [8]:
# Train model
model_trainer = t5_exp.train(id_tokenized_ds, **training_args)

The following columns in the training set don't have a corresponding argument in `T5ForConditionalGeneration.forward` and have been ignored: aspectTerms, sentenceId, aspectCategories, __index_level_0__, text, raw_text. If aspectTerms, sentenceId, aspectCategories, __index_level_0__, text, raw_text are not expected by `T5ForConditionalGeneration.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 3045
  Num Epochs = 4
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 1
  Total optimization steps = 1524


Trainer device: cuda:0

Model training started ....


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,No log,0.303327


Epoch,Training Loss,Validation Loss
1,No log,0.303327
2,0.572600,0.290169


Epoch,Training Loss,Validation Loss
1,No log,0.303327
2,0.572600,0.290169
3,0.282100,0.285414


Epoch,Training Loss,Validation Loss
1,No log,0.303327
2,0.572600,0.290169
3,0.282100,0.285414
4,0.216300,0.288872


Epoch,Training Loss,Validation Loss
1,No log,0.303327
2,0.572600,0.290169
3,0.282100,0.285414
4,0.216300,0.288872


Saving model checkpoint to ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1
Configuration saved in ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/config.json
Model weights saved in ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/pytorch_model.bin
tokenizer config file saved in ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/tokenizer_config.json
Special tokens file saved in ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/special_tokens_map.json
Copy vocab file to ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/spiece.model


## Inference

In [9]:
# Load the data
id_train_file_path = 'Dataset/SemEval14/Train/Laptops_Train.csv'
id_test_file_path = 'Dataset/SemEval14/Test/Laptops_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path, encoding='gbk')

# ood_train_file_path = './Dataset/SemEval14/Train/Restaurants_Train.csv'
# ood_test_file_path = './Dataset/SemEval14/Test/Restaurants_Test.csv'
# ood_tr_df = pd.read_csv(ood_train_file_path)
# ood_te_df = pd.read_csv(ood_test_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set1()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(train_df_id=id_tr_df, test_df_id=id_te_df)  #, train_df_ood=ood_tr_df, test_df_ood=ood_te_df)
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_ate_format(loader.train_df_id, 'term', 'raw_text', 'aspectTerms',
                                                          instruct_handler.ate['bos_instruct1'],
                                                          instruct_handler.ate['eos_instruct'])
if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_ate_format(loader.test_df_id, 'term', 'raw_text', 'aspectTerms',
                                                         instruct_handler.ate['bos_instruct1'],
                                                         instruct_handler.ate['eos_instruct'])

# if loader.train_df_ood is not None:
#     loader.train_df_ood = loader.create_data_in_ate_format(loader.train_df_ood, 'term', 'raw_text', 'aspectTerms', instruct_handler.ate['bos_instruct2'], instruct_handler.ate['eos_instruct'])
# if loader.test_df_ood is not None:
#     loader.test_df_ood = loader.create_data_in_ate_format(loader.test_df_ood, 'term', 'raw_text', 'aspectTerms', instruct_handler.ate['bos_instruct2'], instruct_handler.ate['eos_instruct'])

In [10]:
# Model inference - Loading from Checkpoint
t5_exp = T5Generator(model_out_path)

# Tokenize Datasets
id_ds, id_tokenized_ds, ood_ds, ood_tokenzed_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Get prediction labels - Training set   
id_tr_pred_labels = t5_exp.get_labels(tokenized_dataset=id_tokenized_ds, sample_set='train',
                                      # trained_model_path=model_out_path,
                                      batch_size=16)
id_tr_labels = [i.strip() for i in id_ds['train']['labels']]

# Get prediction labels - Testing set
id_te_pred_labels = t5_exp.get_labels(tokenized_dataset=id_tokenized_ds, sample_set='test',
                                      # trained_model_path=model_out_path,
                                      batch_size=16)
id_te_labels = [i.strip() for i in id_ds['test']['labels']]

Didn't find file ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/added_tokens.json. We won't load it.
loading file ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/spiece.model
loading file ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/tokenizer.json
loading file None
loading file ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/special_tokens_map.json
loading file ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/tokenizer_config.json
loading configuration file ./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1/config.json
Model config T5Config {
  "_name_or_path": "./Models/ate/.modeltk-instruct-base-def-pos-lapt2014_iabsa1",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decod

Map:   0%|          | 0/3045 [00:00<?, ? examples/s]

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Model loaded to:  cuda


100%|██████████| 191/191 [00:43<00:00,  4.37it/s]


Model loaded to:  cuda


100%|██████████| 38/38 [00:08<00:00,  4.40it/s]


In [11]:
p, r, f1, _ = t5_exp.get_metrics(id_tr_labels, id_tr_pred_labels)
print('Train Precision: ', p)
print('Train Recall: ', r)
print('Train F1: ', f1)

p, r, f1, _ = t5_exp.get_metrics(id_te_labels, id_te_pred_labels)
print('Test Precision: ', p)
print('Test Recall: ', r)
print('Test F1: ', f1)

Train Precision:  0.9391438886023724
Train Recall:  0.9302681992337165
Train F1:  0.9346849736943411
Test Precision:  0.9491059147180193
Test Recall:  0.9249329758713136
Test F1:  0.9368635437881874
